# Step 1 — 75/15/15 split + blend + RF\nconvnextv2_base | efficientnetv2_m | deit3_base + mpnet embeddings

In [ ]:
!pip install timm==1.0.3 albumentations sentence-transformers -q
print('OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.1 MB/s eta 0:00:00
OK


In [ ]:
import os, gc, json, random, time, warnings, sys
from pathlib import Path
from collections import deque
import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
warnings.filterwarnings('ignore')
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

PyTorch: 2.11.0+cu128 | GPU: NVIDIA A100-SXM4-40GB


In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB  = 'google.colab' in sys.modules

DEBUG        = False
DEBUG_EPOCHS = 1

if IS_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/food_step1') if not DEBUG else Path('/content/food_step1_debug')
    KAGGLE_DATA = Path('/content/data')
    KAGGLE_DATA.mkdir(parents=True, exist_ok=True)
    if not (KAGGLE_DATA / 'train_labels.csv').exists():
        uploaded = files.upload()
        !mkdir -p ~/.kaggle
        !mv kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        !pip install kaggle -q
        !kaggle competitions download -c m2-food-calorie-estimation -p /content/data
        !unzip -q /content/data/m2-food-calorie-estimation.zip -d /content/data/
    DESC_CSV  = Path('/content/drive/MyDrive/food_descriptions/descriptions_merged.csv')
    EMB_CACHE = Path('/content/drive/MyDrive/food_descriptions/embeddings_mpnet.npz')
elif IS_KAGGLE:
    WORK = Path('/kaggle/working')
    for candidate in [
        Path('/kaggle/input/m2-food-calorie-estimation'),
        Path('/kaggle/input/competitions/m2-food-calorie-estimation'),
    ]:
        if (candidate / 'train_labels.csv').exists():
            KAGGLE_DATA = candidate; break
    else:
        KAGGLE_DATA = list(Path('/kaggle/input').rglob('train_labels.csv'))[0].parent
    DESC_CSV  = Path('/kaggle/input/datasets/yacineabdelouhab/food-descriptions-moondream/descriptions_merged.csv')
    EMB_CACHE = Path('/kaggle/working/embeddings_mpnet.npz')
else:
    WORK      = Path('.')
    KAGGLE_DATA = Path(r'C:\Users\Abdel\.cache\kagglehub\competitions\m2-food-calorie-estimation')
    DESC_CSV  = Path(r'C:\Users\Abdel\Desktop\M2Dauphine\DL_Image\Projet\description_textuel\descriptions_merged.csv')
    EMB_CACHE = WORK / 'embeddings_mpnet.npz'

WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
CKPT_DIR = WORK / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR  = WORK / 'outputs';     OUT_DIR.mkdir(exist_ok=True)

SEED=42; VAL_RATIO=0.15; TEST_RATIO=0.15
PRED_MIN=30.0; PRED_MAX=3600.0
NUM_WORKERS = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, '|', 'debug' if DEBUG else 'normal', '|', WORK)
print('EMB_CACHE:', EMB_CACHE)

Mounted at /content/drive


Saving kaggle.json to kaggle.json
100% 2.06G/2.06G [01:41<00:00, 21.8MB/s]

cuda | normal | /content/drive/MyDrive/food_step1
EMB_CACHE: /content/drive/MyDrive/food_descriptions/embeddings_mpnet.npz


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

TRAIN_IMGS = KAGGLE_DATA / 'train' / 'images'
TEST_IMGS  = KAGGLE_DATA / 'test'  / 'images'

def resolve_image_path(img_dir, filename):
    p = img_dir / filename
    if p.exists(): return str(p)
    alt = p.with_suffix('.png' if p.suffix.lower()=='.jpg' else '.jpg')
    return str(alt) if alt.exists() else str(p)

train_df = pd.read_csv(KAGGLE_DATA / 'train_labels.csv')
test_df  = pd.read_csv(KAGGLE_DATA / 'test_ids.csv')
train_df['path'] = train_df['filename'].apply(lambda f: resolve_image_path(TRAIN_IMGS, f))
test_df['path']  = test_df['filename'].apply(lambda f: resolve_image_path(TEST_IMGS, f))
train_df['target_log'] = np.log1p(train_df['calories'].astype(float))
train_df['cal_bin'] = pd.qcut(train_df['calories'], q=10, labels=False, duplicates='drop')

df_temp, df_test_final = train_test_split(train_df, test_size=TEST_RATIO, random_state=SEED, stratify=train_df['cal_bin'])
df_train, df_val = train_test_split(df_temp, test_size=VAL_RATIO/(1-TEST_RATIO), random_state=SEED, stratify=df_temp['cal_bin'])
df_train      = df_train.reset_index(drop=True)
df_val        = df_val.reset_index(drop=True)
df_test_final = df_test_final.reset_index(drop=True)
print(f'train={len(df_train)} val={len(df_val)} test={len(df_test_final)} kaggle={len(test_df)}')

train=2168 val=465 test=465 kaggle=547


In [ ]:
df_desc = pd.read_csv(DESC_CSV)
descriptions = dict(zip(df_desc['image_id'].astype(str), df_desc['description'].fillna('food dish')))
print(f'{len(descriptions)} descriptions')

3645 descriptions


In [ ]:
from sentence_transformers import SentenceTransformer

if EMB_CACHE.exists():
    tmp = np.load(EMB_CACHE, allow_pickle=True)
    if tmp['vecs'].shape[1] != 768:
        EMB_CACHE.unlink()

if EMB_CACHE.exists():
    saved  = np.load(EMB_CACHE, allow_pickle=True)
    id2emb = {k: saved['vecs'][i] for i, k in enumerate(saved['ids'])}
else:
    st_model = SentenceTransformer('all-mpnet-base-v2', device='cpu')
    ids_list = list(descriptions.keys())
    vecs     = st_model.encode([descriptions[k] for k in ids_list], batch_size=32,
                                show_progress_bar=True, normalize_embeddings=True).astype(np.float32)
    np.savez(EMB_CACHE, ids=np.array(ids_list), vecs=vecs)
    id2emb = {k: vecs[i] for i, k in enumerate(ids_list)}
    del st_model; gc.collect()

TEXT_DIM = next(iter(id2emb.values())).shape[0]
print(f'{len(id2emb)} embeddings x {TEXT_DIM}-dim')

3645 embeddings x 768-dim


In [ ]:
class MultimodalFoodDataset(Dataset):
    def __init__(self, df, transforms, id2emb, is_train=True):
        self.df         = df.reset_index(drop=True)
        self.transforms = transforms
        self.id2emb     = id2emb
        self.is_train   = is_train
        self.fallback   = np.zeros(TEXT_DIM, dtype=np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.path, cv2.IMREAD_COLOR)
        if img is None: raise FileNotFoundError(row.path)
        img      = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img      = self.transforms(image=img)['image']
        text_emb = torch.tensor(self.id2emb.get(str(row.image_id), self.fallback), dtype=torch.float32)
        if self.is_train:
            return img, text_emb, torch.tensor(float(row.target_log), dtype=torch.float32)
        return img, text_emb, str(row.image_id)

def make_loader(df, transforms, batch_size, is_train, shuffle, drop_last=False):
    ds = MultimodalFoodDataset(df, transforms, id2emb, is_train=is_train)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=drop_last)

In [ ]:
BACKBONES_WITH_IMG_SIZE = ('swin', 'vit', 'deit', 'beit')

class MultimodalRegressor(nn.Module):
    def __init__(self, backbone_name, img_size, text_dim, dropout=0.25):
        super().__init__()
        needs_img_size = any(k in backbone_name for k in BACKBONES_WITH_IMG_SIZE)
        extra = {'img_size': img_size} if needs_img_size else {}
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, global_pool='avg', **extra)
        fused_dim = self.backbone.num_features + text_dim
        self.head = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 128),       nn.GELU(), nn.Dropout(dropout / 2),
            nn.Linear(128, 1),
        )
        print(f'  fused={fused_dim}')

    def forward(self, img, text_emb):
        return self.head(torch.cat([self.backbone(img), text_emb], dim=1)).squeeze(1)

def get_optimizer_llrd(model, base_lr, wd, n_groups=4, decay=0.25):
    bp = [(n,p) for n,p in model.backbone.named_parameters() if p.requires_grad]
    hp = [(n,p) for n,p in model.head.named_parameters()    if p.requires_grad]
    gs = max(1, len(bp)//n_groups); pg = []
    for g in range(n_groups):
        s = g*gs; e = (g+1)*gs if g < n_groups-1 else len(bp)
        lr_g = base_lr * (decay**(n_groups-1-g)); chunk = bp[s:e]
        wd_  = [p for n,p in chunk if p.ndim>1 and not n.endswith('bias') and 'norm' not in n.lower()]
        nwd  = [p for n,p in chunk if p.ndim<=1 or n.endswith('bias') or 'norm' in n.lower()]
        if wd_:  pg.append({'params':wd_,  'lr':lr_g, 'weight_decay':wd})
        if nwd:  pg.append({'params':nwd,  'lr':lr_g, 'weight_decay':0.})
    hwd  = [p for n,p in hp if p.ndim>1 and not n.endswith('bias') and 'norm' not in n.lower()]
    hnwd = [p for n,p in hp if p.ndim<=1 or n.endswith('bias') or 'norm' in n.lower()]
    if hwd:  pg.append({'params':hwd,  'lr':base_lr, 'weight_decay':wd})
    if hnwd: pg.append({'params':hnwd, 'lr':base_lr, 'weight_decay':0.})
    return torch.optim.AdamW(pg)

In [ ]:
def preds_to_calories(p): return np.clip(np.expm1(p), PRED_MIN, PRED_MAX)

MIXUP_ALPHA = 0.4; AUG_MIX_PROB = 0.3

def mixup(imgs, texts, targets):
    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam*imgs+(1-lam)*imgs[idx], lam*texts+(1-lam)*texts[idx], lam*targets+(1-lam)*targets[idx]

def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train(); total = 0.; optimizer.zero_grad(set_to_none=True)
    for imgs, texts, targets in loader:
        imgs    = imgs.to(device, non_blocking=True)
        texts   = texts.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if random.random() < AUG_MIX_PROB:
            imgs, texts, targets = mixup(imgs, texts, targets)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            loss = criterion(model(imgs, texts), targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        if scheduler: scheduler.step()
        total += float(loss.item())
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); losses, preds = [], []
    for imgs, texts, targets in loader:
        imgs    = imgs.to(device, non_blocking=True)
        texts   = texts.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            out = model(imgs, texts); loss = criterion(out, targets)
        losses.append(float(loss.item())); preds.append(out.float().cpu().numpy())
    return float(np.mean(losses)), np.concatenate(preds)

@torch.no_grad()
def predict(model, loader):
    model.eval(); all_ids, all_preds = [], []
    for imgs, texts, ids in loader:
        imgs  = imgs.to(device, non_blocking=True)
        texts = texts.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            out      = model(imgs, texts)
            out_flip = model(torch.flip(imgs, dims=[3]), texts)
        all_preds.append(((out+out_flip)/2).float().cpu().numpy())
        all_ids.extend(ids if isinstance(ids, list) else list(ids))
    return all_ids, np.concatenate(all_preds)

In [ ]:
MODELS = [
    ('convnextv2_base.fcmae_ft_in22k_in1k',        224, 'convnextv2_base_mm',  64),
    ('tf_efficientnetv2_m.in21k_ft_in1k',           224, 'efficientnetv2_m_mm', 64),
    ('deit3_base_patch16_224.fb_in22k_ft_in1k',     224, 'deit3_base_mm',       64),
]

LR=2e-4; WD=1e-2; DROPOUT=0.25
EPOCHS    = DEBUG_EPOCHS if DEBUG else 100
MA_WINDOW = 1  if DEBUG else 10
PATIENCE  = 1  if DEBUG else 15

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f'VRAM libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB')

VRAM libre: 42.0 GB / 42.4 GB


In [ ]:
results = {}

for backbone_name, img_size, exp_name, bs in MODELS:
    print(f'\n{exp_name}')
    ckpt_path  = CKPT_DIR / f'{exp_name}_best.pth'
    preds_path = OUT_DIR  / f'{exp_name}_preds.npz'

    if not DEBUG and ckpt_path.exists() and preds_path.exists():
        saved = np.load(preds_path)
        results[exp_name] = {
            'val_mae':   mean_absolute_error(df_val['calories'].values,        preds_to_calories(saved['val_preds'])),
            'test_mae':  mean_absolute_error(df_test_final['calories'].values, preds_to_calories(saved['test_final_preds'])),
            'best_epoch': int(torch.load(ckpt_path, map_location='cpu', weights_only=False)['epoch']),
        }
        print(f'  skip | val={results[exp_name]["val_mae"]:.2f} test={results[exp_name]["test_mae"]:.2f} epoch={results[exp_name]["best_epoch"]}')
        continue

    train_tfm = A.Compose([
        A.RandomResizedCrop(size=(img_size,img_size), scale=(0.65,1.0), ratio=(0.80,1.20), interpolation=cv2.INTER_AREA),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08, p=0.6),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])
    val_tfm = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

    train_loader = make_loader(df_train, train_tfm, bs, is_train=True,  shuffle=True,  drop_last=True)
    val_loader   = make_loader(df_val,   val_tfm,   bs, is_train=True,  shuffle=False)

    model     = MultimodalRegressor(backbone_name, img_size, TEXT_DIM, DROPOUT).to(device)
    optimizer = get_optimizer_llrd(model, base_lr=LR, wd=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS*len(train_loader), eta_min=1e-7)
    scaler    = GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.HuberLoss(delta=0.5)

    best_ma = float('inf'); best_mae = float('inf'); wait = 0
    mae_history = deque(maxlen=MA_WINDOW)

    for epoch in range(1, EPOCHS+1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)
        _, val_log = evaluate(model, val_loader, criterion)
        val_mae    = mean_absolute_error(df_val['calories'].values, preds_to_calories(val_log))
        mae_history.append(val_mae)
        ma = float(np.mean(mae_history))

        if val_mae < best_mae:
            best_mae = val_mae
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'best_mae': best_mae, 'backbone': backbone_name, 'img_size': img_size}, ckpt_path)

        print(f'  {epoch:03d}/{EPOCHS} | train={train_loss:.4f} val={val_mae:.2f} ma={ma:.2f} best={best_mae:.2f} | {time.time()-t0:.0f}s', flush=True)

        if len(mae_history) == MA_WINDOW:
            if ma < best_ma: best_ma = ma; wait = 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    print(f'  early stop ep={epoch} best={best_mae:.2f}'); break

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])

    test_loader   = make_loader(df_test_final, val_tfm, bs, is_train=True,  shuffle=False)
    kaggle_loader = make_loader(test_df,       val_tfm, bs, is_train=False, shuffle=False)

    _, vpl = evaluate(model, val_loader,  criterion)
    _, tpl = evaluate(model, test_loader, criterion)
    kids, kpl = predict(model, kaggle_loader)
    np.savez(preds_path, val_preds=vpl, test_final_preds=tpl,
             kaggle_ids=np.array(kids), kaggle_preds=preds_to_calories(kpl))

    results[exp_name] = {
        'val_mae':   mean_absolute_error(df_val['calories'].values,        preds_to_calories(vpl)),
        'test_mae':  mean_absolute_error(df_test_final['calories'].values, preds_to_calories(tpl)),
        'best_epoch': ckpt['epoch'],
    }
    print(f'  val={results[exp_name]["val_mae"]:.2f} test={results[exp_name]["test_mae"]:.2f} epoch={ckpt["epoch"]}')

    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


convnextv2_base_mm


model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

  fused=1792
  001/100 | train=0.5645 val=193.10 ma=193.10 best=193.10 | 90s
  002/100 | train=0.1368 val=178.53 ma=185.81 best=178.53 | 53s
  003/100 | train=0.1063 val=177.01 ma=182.88 best=177.01 | 53s
  004/100 | train=0.1121 val=110.17 ma=164.70 best=110.17 | 56s
  005/100 | train=0.0991 val=117.11 ma=155.18 best=110.17 | 54s
  006/100 | train=0.0989 val=138.82 ma=152.46 best=110.17 | 54s
  007/100 | train=0.1069 val=116.94 ma=147.38 best=110.17 | 53s
  008/100 | train=0.1004 val=118.15 ma=143.73 best=110.17 | 54s
  009/100 | train=0.0818 val=154.33 ma=144.91 best=110.17 | 55s
  010/100 | train=0.0813 val=103.10 ma=140.73 best=103.10 | 54s
  011/100 | train=0.0775 val=108.63 ma=132.28 best=103.10 | 54s
  012/100 | train=0.0797 val=89.05 ma=123.33 best=89.05 | 53s
  013/100 | train=0.0695 val=120.35 ma=117.66 best=89.05 | 53s
  014/100 | train=0.0769 val=138.21 ma=120.47 best=89.05 | 53s
  015/100 | train=0.0676 val=124.36 ma=121.19 best=89.05 | 56s
  016/100 | train=0.0750 val=107

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  fused=1536
  001/100 | train=0.5883 val=170.75 ma=170.75 best=170.75 | 53s
  002/100 | train=0.1614 val=179.16 ma=174.95 best=170.75 | 54s
  003/100 | train=0.1237 val=119.11 ma=156.34 best=119.11 | 54s
  004/100 | train=0.1215 val=108.71 ma=144.43 best=108.71 | 56s
  005/100 | train=0.1062 val=161.72 ma=147.89 best=108.71 | 53s
  006/100 | train=0.0983 val=128.95 ma=144.73 best=108.71 | 52s
  007/100 | train=0.1083 val=164.30 ma=147.53 best=108.71 | 54s
  008/100 | train=0.0882 val=151.48 ma=148.02 best=108.71 | 54s
  009/100 | train=0.0901 val=127.34 ma=145.72 best=108.71 | 54s
  010/100 | train=0.0821 val=101.04 ma=141.26 best=101.04 | 53s
  011/100 | train=0.1077 val=101.65 ma=134.35 best=101.04 | 53s
  012/100 | train=0.0891 val=113.14 ma=127.74 best=101.04 | 52s
  013/100 | train=0.0695 val=107.49 ma=126.58 best=101.04 | 54s
  014/100 | train=0.0908 val=147.91 ma=130.50 best=101.04 | 52s
  015/100 | train=0.0684 val=118.53 ma=126.18 best=101.04 | 53s
  016/100 | train=0.0691 va

In [ ]:
exp_name  = 'convnextv2_base_augmax'
backbone  = 'convnextv2_base.fcmae_ft_in22k_in1k'
img_size  = 224; bs = 64
ckpt_path  = CKPT_DIR / f'{exp_name}_best.pth'
preds_path = OUT_DIR  / f'{exp_name}_preds.npz'
print(f'\n{exp_name}')

if not DEBUG and ckpt_path.exists() and preds_path.exists():
    saved = np.load(preds_path)
    results[exp_name] = {
        'val_mae':   mean_absolute_error(df_val['calories'].values,        preds_to_calories(saved['val_preds'])),
        'test_mae':  mean_absolute_error(df_test_final['calories'].values, preds_to_calories(saved['test_final_preds'])),
        'best_epoch': int(torch.load(ckpt_path, map_location='cpu', weights_only=False)['epoch']),
    }
    print(f'  skip | val={results[exp_name]["val_mae"]:.2f} test={results[exp_name]["test_mae"]:.2f} epoch={results[exp_name]["best_epoch"]}')
else:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    class _ImgDS(Dataset):
        def __init__(self, df, tfm, is_train=True):
            self.df = df.reset_index(drop=True); self.tfm = tfm; self.is_train = is_train
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = cv2.cvtColor(cv2.imread(row.path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            img = self.tfm(image=img)['image']
            return (img, torch.tensor(float(row.target_log), dtype=torch.float32)) if self.is_train else (img, str(row.image_id))

    def _make(df, tfm, is_train, shuffle, drop_last=False):
        return DataLoader(_ImgDS(df, tfm, is_train), batch_size=bs, shuffle=shuffle,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=drop_last)

    class _Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0, global_pool='avg')
            d = self.backbone.num_features
            self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d,512), nn.GELU(), nn.Dropout(DROPOUT),
                                      nn.Linear(512,128), nn.GELU(), nn.Dropout(DROPOUT/2), nn.Linear(128,1))
            print(f'  features={d}')
        def forward(self, x): return self.head(self.backbone(x)).squeeze(1)

    train_tfm = A.Compose([
        A.RandomResizedCrop(size=(img_size,img_size), scale=(0.5,1.0), ratio=(0.75,1.33), interpolation=cv2.INTER_AREA),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2), A.RandomRotate90(p=0.3),
        A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
        A.GaussianBlur(blur_limit=(3,7), p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.3),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])
    val_tfm = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])

    tr_l = _make(df_train, train_tfm, True,  True,  True)
    va_l = _make(df_val,   val_tfm,   True,  False)

    model     = _Model().to(device)
    bp = [(n,p) for n,p in model.backbone.named_parameters() if p.requires_grad]
    hp = [(n,p) for n,p in model.head.named_parameters()    if p.requires_grad]
    gs = max(1, len(bp)//4); pg = []
    for g in range(4):
        chunk = bp[g*gs:(g+1)*gs if g<3 else len(bp)]
        lr_g  = LR * (0.25**(3-g))
        wd_  = [p for n,p in chunk if p.ndim>1 and 'bias' not in n and 'norm' not in n.lower()]
        nwd  = [p for n,p in chunk if not (p.ndim>1 and 'bias' not in n and 'norm' not in n.lower())]
        if wd_:  pg.append({'params':wd_,  'lr':lr_g, 'weight_decay':WD})
        if nwd:  pg.append({'params':nwd,  'lr':lr_g, 'weight_decay':0.})
    hwd  = [p for n,p in hp if p.ndim>1 and 'bias' not in n and 'norm' not in n.lower()]
    hnwd = [p for n,p in hp if not (p.ndim>1 and 'bias' not in n and 'norm' not in n.lower())]
    if hwd:  pg.append({'params':hwd,  'lr':LR, 'weight_decay':WD})
    if hnwd: pg.append({'params':hnwd, 'lr':LR, 'weight_decay':0.})
    optimizer = torch.optim.AdamW(pg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS*len(tr_l), eta_min=1e-7)
    scaler    = GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.HuberLoss(delta=0.5)

    best_ma = float('inf'); best_mae = float('inf'); wait = 0
    mae_history = deque(maxlen=MA_WINDOW)

    for epoch in range(1, EPOCHS+1):
        t0 = time.time(); model.train(); total = 0.; optimizer.zero_grad(set_to_none=True)
        for imgs, targets in tr_l:
            imgs = imgs.to(device, non_blocking=True); targets = targets.to(device, non_blocking=True)
            if random.random() < AUG_MIX_PROB:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA); idx = torch.randperm(imgs.size(0), device=imgs.device)
                imgs = lam*imgs+(1-lam)*imgs[idx]; targets = lam*targets+(1-lam)*targets[idx]
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                loss = criterion(model(imgs), targets)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            if scheduler: scheduler.step()
            total += float(loss.item())
        train_loss = total / len(tr_l)

        model.eval(); vp = []
        with torch.no_grad():
            for imgs, targets in va_l:
                imgs = imgs.to(device, non_blocking=True)
                with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                    vp.append(model(imgs).float().cpu().numpy())
        val_log = np.concatenate(vp)
        val_mae = mean_absolute_error(df_val['calories'].values, preds_to_calories(val_log))
        mae_history.append(val_mae); ma = float(np.mean(mae_history))

        if val_mae < best_mae:
            best_mae = val_mae
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'best_mae': best_mae, 'backbone': backbone, 'img_size': img_size}, ckpt_path)

        print(f'  {epoch:03d}/{EPOCHS} | train={train_loss:.4f} val={val_mae:.2f} ma={ma:.2f} best={best_mae:.2f} | {time.time()-t0:.0f}s', flush=True)

        if len(mae_history) == MA_WINDOW:
            if ma < best_ma: best_ma = ma; wait = 0
            else:
                wait += 1
                if wait >= PATIENCE: print(f'  early stop ep={epoch} best={best_mae:.2f}'); break

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state']); model.eval()

    te_l = _make(df_test_final, val_tfm, True,  False)
    ka_l = _make(test_df,       val_tfm, False, False)

    vp2 = []; tp2 = []
    with torch.no_grad():
        for imgs, targets in va_l:
            imgs = imgs.to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                vp2.append(model(imgs).float().cpu().numpy())
        for imgs, targets in te_l:
            imgs = imgs.to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                tp2.append(model(imgs).float().cpu().numpy())
    vpl = np.concatenate(vp2); tpl = np.concatenate(tp2)

    all_ids2 = []; all_p2 = []
    with torch.no_grad():
        for imgs, ids in ka_l:
            imgs = imgs.to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                out = model(imgs); out_flip = model(torch.flip(imgs, dims=[3]))
            all_p2.append(((out+out_flip)/2).float().cpu().numpy())
            all_ids2.extend(ids if isinstance(ids, list) else list(ids))
    kpl = np.concatenate(all_p2)

    np.savez(preds_path, val_preds=vpl, test_final_preds=tpl,
             kaggle_ids=np.array(all_ids2), kaggle_preds=preds_to_calories(kpl))

    results[exp_name] = {
        'val_mae':   mean_absolute_error(df_val['calories'].values,        preds_to_calories(vpl)),
        'test_mae':  mean_absolute_error(df_test_final['calories'].values, preds_to_calories(tpl)),
        'best_epoch': ckpt['epoch'],
    }
    print(f'  val={results[exp_name]["val_mae"]:.2f} test={results[exp_name]["test_mae"]:.2f} epoch={ckpt["epoch"]}')
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


convnextv2_base_augmax
  features=1024
  001/100 | train=0.7226 val=203.05 ma=203.05 best=203.05 | 54s
  002/100 | train=0.1643 val=161.93 ma=182.49 best=161.93 | 55s
  003/100 | train=0.1364 val=169.79 ma=178.26 best=161.93 | 52s
  004/100 | train=0.1187 val=134.43 ma=167.30 best=134.43 | 53s
  005/100 | train=0.1048 val=146.29 ma=163.10 best=134.43 | 55s
  006/100 | train=0.1016 val=120.00 ma=155.92 best=120.00 | 54s
  007/100 | train=0.1091 val=133.46 ma=152.71 best=120.00 | 54s
  008/100 | train=0.0952 val=122.85 ma=148.98 best=120.00 | 53s
  009/100 | train=0.1066 val=111.28 ma=144.79 best=111.28 | 53s
  010/100 | train=0.0921 val=182.47 ma=148.56 best=111.28 | 54s
  011/100 | train=0.0965 val=128.98 ma=141.15 best=111.28 | 52s
  012/100 | train=0.0865 val=104.49 ma=135.40 best=104.49 | 55s
  013/100 | train=0.0823 val=104.02 ma=128.83 best=104.02 | 55s
  014/100 | train=0.0859 val=117.20 ma=127.10 best=104.02 | 53s
  015/100 | train=0.0975 val=126.38 ma=125.11 best=104.02 | 52s


In [ ]:
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor

npz_files   = sorted(OUT_DIR.glob('*_preds.npz'))
model_names = []; val_preds_l = []; test_preds_l = []; kag_preds_l = []; kaggle_ids = None

for f in npz_files:
    data = np.load(f, allow_pickle=True)
    vp = preds_to_calories(data['val_preds'])
    tp = preds_to_calories(data['test_final_preds'])
    kp = data['kaggle_preds']
    val_preds_l.append(vp); test_preds_l.append(tp); kag_preds_l.append(kp)
    if kaggle_ids is None: kaggle_ids = list(data['kaggle_ids'])
    model_names.append(f.stem.replace('_preds',''))
    print(f'  {f.stem:<35} val={mean_absolute_error(df_val["calories"].values, vp):.2f} test={mean_absolute_error(df_test_final["calories"].values, tp):.2f}')

val_preds  = np.stack(val_preds_l)
test_preds = np.stack(test_preds_l)
kag_preds  = np.stack(kag_preds_l)
y_val  = df_val['calories'].values
y_test = df_test_final['calories'].values
M      = len(model_names)

  convnextv2_base_augmax_preds        val=61.94 test=64.62
  convnextv2_base_mm_preds            val=72.89 test=54.14
  deit3_base_mm_preds                 val=86.41 test=84.63
  efficientnetv2_m_mm_preds           val=116.62 test=109.67


In [ ]:
def blend_mae(w):
    w = np.abs(w); w = w / w.sum()
    return mean_absolute_error(y_val, (val_preds * w[:, None]).sum(axis=0))

res  = minimize(blend_mae, np.ones(M)/M, method='Nelder-Mead',
                options={'maxiter': 10000, 'xatol': 1e-6, 'fatol': 1e-6})
w_nm = np.abs(res.x); w_nm /= w_nm.sum()

blend_val_nm  = (val_preds  * w_nm[:, None]).sum(axis=0)
blend_test_nm = (test_preds * w_nm[:, None]).sum(axis=0)
blend_kag_nm  = (kag_preds  * w_nm[:, None]).sum(axis=0)
mae_val_nm    = mean_absolute_error(y_val,  blend_val_nm)
mae_test_nm   = mean_absolute_error(y_test, blend_test_nm)

for name, w in zip(model_names, w_nm):
    print(f'  {name:<35} {w:.4f}')
print(f'NM  val={mae_val_nm:.2f} test={mae_test_nm:.2f}')

  convnextv2_base_augmax              0.8265
  convnextv2_base_mm                  0.0679
  deit3_base_mm                       0.1055
  efficientnetv2_m_mm                 0.0000
NM  val=61.00 test=62.02


In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=SEED, n_jobs=-1)
rf.fit(val_preds.T, y_val)

blend_val_rf  = rf.predict(val_preds.T)
blend_test_rf = rf.predict(test_preds.T)
blend_kag_rf  = rf.predict(kag_preds.T)
mae_val_rf    = mean_absolute_error(y_val,  blend_val_rf)
mae_test_rf   = mean_absolute_error(y_test, blend_test_rf)

print(f'RF  val={mae_val_rf:.2f} test={mae_test_rf:.2f}')
for name, imp in sorted(zip(model_names, rf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:<35} {imp:.4f}')

RF  val=55.50 test=69.44
  convnextv2_base_augmax              0.7540
  convnextv2_base_mm                  0.1485
  deit3_base_mm                       0.0829
  efficientnetv2_m_mm                 0.0146


In [ ]:
print(f'{"model":<35} {"val":>8} {"test":>8}')
for name, r in results.items():
    print(f'{name:<35} {r["val_mae"]:>8.2f} {r["test_mae"]:>8.2f}  (ep {r["best_epoch"]})')
print(f'{"NM blend":<35} {mae_val_nm:>8.2f} {mae_test_nm:>8.2f}')
print(f'{"RF blend":<35} {mae_val_rf:>8.2f} {mae_test_rf:>8.2f}')

if mae_test_nm <= mae_test_rf:
    best_kag = blend_kag_nm; best_tag = f'nm_{mae_test_nm:.2f}'
else:
    best_kag = blend_kag_rf; best_tag = f'rf_{mae_test_rf:.2f}'

sub = pd.DataFrame({'image_id': kaggle_ids, 'calories': np.clip(best_kag, PRED_MIN, PRED_MAX).round(2)})
sub.to_csv(WORK / f'submission_{best_tag}.csv', index=False)
print(f'\nsubmission_{best_tag}.csv')
print(sub.head())

if not DEBUG:
    best_epochs = {name: r['best_epoch'] for name, r in results.items()}
    nm_weights  = {name: float(w) for name, w in zip(model_names, w_nm)}
    (WORK / 'best_epochs.json').write_text(json.dumps(best_epochs, indent=2))
    (WORK / 'nelder_mead_weights.json').write_text(json.dumps(nm_weights, indent=2))
    print('best_epochs.json:', best_epochs)

model                                    val     test
convnextv2_base_mm                     72.89    54.14  (ep 31)
efficientnetv2_m_mm                   116.62   109.67  (ep 65)
deit3_base_mm                          86.41    84.63  (ep 32)
convnextv2_base_augmax                 61.94    64.62  (ep 61)
NM blend                               61.00    62.02
RF blend                               55.50    69.44

submission_nm_62.02.csv
    image_id  calories
0  test_0000     84.12
1  test_0001    152.53
2  test_0002    149.71
3  test_0003    263.50
4  test_0004    204.20
best_epochs.json: {'convnextv2_base_mm': 31, 'efficientnetv2_m_mm': 65, 'deit3_base_mm': 32, 'convnextv2_base_augmax': 61}


In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
import numpy as np
from pathlib import Path
from google.colab import files

OUT_BASE = Path('/content/drive/MyDrive/food_step1/outputs')
OUT_TINY = Path('/content/drive/MyDrive/food_step1_tiny/outputs')

PRED_MIN=30.0; PRED_MAX=3600.0
def ptc(p): return np.clip(np.expm1(p), PRED_MIN, PRED_MAX)

all_names = []; val_l = []; test_l = []; kag_l = []; kaggle_ids = None

for out_dir in [OUT_BASE, OUT_TINY]:
    for f in sorted(out_dir.glob('*_preds.npz')):
        data = np.load(f, allow_pickle=True)
        vp = ptc(data['val_preds'])
        tp = ptc(data['test_final_preds'])
        kp = data['kaggle_preds']
        val_l.append(vp); test_l.append(tp); kag_l.append(kp)
        if kaggle_ids is None: kaggle_ids = list(data['kaggle_ids'])
        name = f'{out_dir.parent.name}/{f.stem.replace("_preds","")}'
        all_names.append(name)
        print(f'  {name:<50} val={mean_absolute_error(df_val["calories"].values, vp):.2f} test={mean_absolute_error(df_test_final["calories"].values, tp):.2f}')

val_all  = np.stack(val_l)
test_all = np.stack(test_l)
kag_all  = np.stack(kag_l)
y_val    = df_val['calories'].values
y_test   = df_test_final['calories'].values
M        = len(all_names)

# NM
def blend_mae(w):
    w = np.abs(w); w /= w.sum()
    return mean_absolute_error(y_val, (val_all * w[:,None]).sum(axis=0))
res  = minimize(blend_mae, np.ones(M)/M, method='Nelder-Mead', options={'maxiter':20000,'xatol':1e-6,'fatol':1e-6})
w_nm = np.abs(res.x); w_nm /= w_nm.sum()
blend_kag_nm  = (kag_all * w_nm[:,None]).sum(axis=0)
mae_val_nm    = mean_absolute_error(y_val,  (val_all  * w_nm[:,None]).sum(axis=0))
mae_test_nm   = mean_absolute_error(y_test, (test_all * w_nm[:,None]).sum(axis=0))

# RF
rf = RandomForestRegressor(n_estimators=500, max_depth=2, random_state=42, n_jobs=-1)
rf.fit(val_all.T, y_val)
mae_val_rf   = mean_absolute_error(y_val,  rf.predict(val_all.T))
mae_test_rf  = mean_absolute_error(y_test, rf.predict(test_all.T))
blend_kag_rf = rf.predict(kag_all.T)

# DT
dt = DecisionTreeRegressor(max_depth=2, random_state=42)
dt.fit(val_all.T, y_val)
mae_val_dt   = mean_absolute_error(y_val,  dt.predict(val_all.T))
mae_test_dt  = mean_absolute_error(y_test, dt.predict(test_all.T))
blend_kag_dt = dt.predict(kag_all.T)

# Poly+Ridge
poly = PolynomialFeatures(degree=2, include_bias=False)
X_val_p  = poly.fit_transform(val_all.T)
X_test_p = poly.transform(test_all.T)
X_kag_p  = poly.transform(kag_all.T)
pr = Ridge(alpha=1.0)
pr.fit(X_val_p, y_val)
mae_val_poly   = mean_absolute_error(y_val,  pr.predict(X_val_p))
mae_test_poly  = mean_absolute_error(y_test, pr.predict(X_test_p))
blend_kag_poly = pr.predict(X_kag_p)

print(f'\n{"method":<20} {"val":>8} {"test":>8}')
print(f'{"NM":<20} {mae_val_nm:>8.2f} {mae_test_nm:>8.2f}')
print(f'{"RF (depth=2)":<20} {mae_val_rf:>8.2f} {mae_test_rf:>8.2f}')
print(f'{"DT (depth=2)":<20} {mae_val_dt:>8.2f} {mae_test_dt:>8.2f}')
print(f'{"Poly+Ridge":<20} {mae_val_poly:>8.2f} {mae_test_poly:>8.2f}')

print(f'\nNM weights:')
for name, w in sorted(zip(all_names, w_nm), key=lambda x: -x[1]):
    print(f'  {name:<50} {w:.4f}')

results_blend = {
    'nm':   (mae_test_nm,   blend_kag_nm),
    'rf':   (mae_test_rf,   blend_kag_rf),
    'dt':   (mae_test_dt,   blend_kag_dt),
    'poly': (mae_test_poly, blend_kag_poly),
}
best_name = min(results_blend, key=lambda k: results_blend[k][0])
best_preds = results_blend[best_name][1]
print(f'\nbest: {best_name} (test={results_blend[best_name][0]:.2f})')

sub_path = f'/content/drive/MyDrive/food_step1/submission_all_best.csv'
sub = pd.DataFrame({'image_id': kaggle_ids, 'calories': np.clip(best_preds, PRED_MIN, PRED_MAX).round(2)})
sub.to_csv(sub_path, index=False)
files.download(sub_path)

  food_step1/convnextv2_base_augmax                  val=61.94 test=64.62
  food_step1/convnextv2_base_mm                      val=72.89 test=54.14
  food_step1/deit3_base_mm                           val=86.41 test=84.63
  food_step1/efficientnetv2_m_mm                     val=116.62 test=109.67
  food_step1_tiny/convnextv2_tiny_mm                 val=91.20 test=79.51
  food_step1_tiny/deit3_small_mm                     val=96.62 test=84.50
  food_step1_tiny/efficientnetv2_s_mm                val=125.09 test=131.35

method                    val     test
NM                      61.58    61.79
RF (depth=2)           108.00   115.46
DT (depth=2)           121.43   131.53
Poly+Ridge              49.22   115.22

NM weights:
  food_step1/convnextv2_base_augmax                  0.6738
  food_step1/deit3_base_mm                           0.2273
  food_step1/convnextv2_base_mm                      0.0820
  food_step1_tiny/efficientnetv2_s_mm                0.0113
  food_step1/efficientnetv2_m

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from pathlib import Path
print(json.loads((Path('/content/drive/MyDrive/food_step1') / 'best_epochs.json').read_text()))

{'convnextv2_base_mm': 31, 'efficientnetv2_m_mm': 65, 'deit3_base_mm': 32, 'convnextv2_base_augmax': 61}
